# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` data curation library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) JSON-LD file located at:
<br>**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant --quiet


## 1. Data Loading
Load dataset metadata and prepare for record exploration using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Initialize the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Display dataset-level metadata
meta = dataset.metadata
print(f"{getattr(meta, 'name', 'Unknown dataset')}: {getattr(meta, 'description', '')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")
print(f"URL: {getattr(meta, '@id', croissant_url)}")

## 2. Data Overview
Let's enumerate the available record sets in the dataset, along with their fields/columns, referencing each entity by its `@id` field as recommended for Croissant datasets.

In [ ]:
# Discover record sets and their fields (@id is the Croissant idiomatic identifier)
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.")
print("\nRecord Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    if 'field' in rs:
        print("  Fields/Columns ({})".format(len(rs['field'])))
        for field in rs['field']:
            if isinstance(field, dict):
                fid = field.get('@id')
                fname = field.get('name', 'N/A')
            else:  # Sometimes only @id is listed
                fid = field
                fname = 'N/A'
            print(f"    - {fid}: {fname}")
    else:
        print("  (No fields listed)")
    print()


## 3. Data Extraction
Let's extract data from one or more record sets. We'll use `@id` values discovered above and load their data as pandas DataFrames for analysis.

Replace `<your_record_set_id>` in the code with the actual `@id` of a record set you'd like to explore. All record set IDs used as variables must match those discovered above!

In [ ]:
# Specify the record set(s) you wish to load, by @id. For demonstration,
# we attempt to load the first discovered record set (edit as needed).
record_set_ids = [rs["@id"] for rs in record_sets]

# We'll preview records and load as DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Reading records from: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            print(f"  Loaded {len(df)} rows. Columns (by @id):\n  {list(df.columns)}\n")
            dataframes[record_set_id] = df
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Could not load records. {e}")

# Preview the first few records of the main record set (if available)
main_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_rs_id and main_rs_id in dataframes:
    print(f"\nPreview of the first 5 records from record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some typical preprocessing and exploration steps:
- Filter: Select records where a numeric field (e.g. coverage, peptide_count, MW) exceeds a threshold
- Normalize: Standardize a numeric field
- Group: Aggregate by a field (such as description, sample type or other categorical field)
<br>These are examples. Please adjust the `numeric_field_id` and `group_field_id` for your actual use case found in the header cell above.

**Remember**: All fields must be referenced by their `@id`, not display names.

In [ ]:
# Define which record set and fields to analyze (edit for your dataset's specific fields @ids)

record_set_id = main_rs_id  # Use the primary record set discovered above.
df = dataframes.get(record_set_id)

if df is not None and not df.empty:
    # Try to select a numeric field by @id
    candidates = [col for col in df.columns if df[col].dtype in ['int64','float64'] or pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields detected (by @id): {candidates}")

    numeric_field = candidates[0] if candidates else None
    threshold = 10

    if numeric_field is not None:
        # Filter records for this field above a threshold
        print(f"Filtering {record_set_id} where {numeric_field} > {threshold}")
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered {len(filtered_df)} records.")
        display(filtered_df.head())
        # Normalize the numeric field
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - mean) / std if std > 0 else filtered_df[numeric_field]
        print(f"\nNormalized {numeric_field}:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
        # Group by another field, if available
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        group_field = group_candidates[0] if group_candidates else None
        if group_field is not None:
            print(f"\nGrouping by {group_field}:")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df[[numeric_field, numeric_field + '_normalized']].head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields available for this record set.")
else:
    print("No records available for analysis.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and, if possible, how it varies by one of the group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # If a group field exists, show boxplot
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you've:
- Loaded the Croissant metadata and explored the dataset structure (`@id`-driven referencing)
- Discovered available record sets and fields
- Extracted records as DataFrames for programmatic data analysis
- Performed basic EDA: filtering, normalization, and grouping by field
- Visualized the distribution and relationships among selected variables

**Next Steps:**
You can adapt the filters, normalization, and groupings to the domain specifics of this protein mass spectrometry data. Consult the Croissant metadata JSON-LD for more details on the meaning of each `@id` and see [mlcroissant documentation](https://mlcommons.org/croissant/) for further examples. Happy exploring!
